# VLM Sınav Eval Notebook

Bu notebook, Google Drive'daki YKS sınav soru resimlerini okuyarak birden fazla vision-language modeline batch inference ile sorar, cevapları parse eder ve detaylı raporlama üretir.

**Kullanım:**
1. `Konfigurasyon` hücresinde model listesini ve Drive yolunu ayarla
2. Tüm hücreleri sırasıyla çalıştır
3. Sonuçlar Drive'daki `OUTPUT_DIR` klasörüne JSON + CSV olarak kaydedilir

**Kesintiden devam:**
Colab kesilirse endişelenme — tüm sonuçlar anlık olarak Drive'a kaydedilir. Notebook'u tekrar çalıştırman yeterli; tamamlanan modeller otomatik atlanır, kaldığı yerden devam eder. Sıfırdan başlamak istersen `checkpoint.json` dosyasını sil.

In [ ]:
# ============================================================
# HUCRE 1: KURULUM
# ============================================================
!pip install -q vllm transformers Pillow pandas tabulate matplotlib seaborn tqdm

In [ ]:
# ============================================================
# HUCRE 2: KONFIGURASYON
# Bu hucreyi ihtiyacina gore duzenle.
# ============================================================

# ----------------------------------------------------------
# MODEL LISTESI
# Istedigin HuggingFace vision-language modelini ekle/cikar.
# Her satir bir dict: en az "name" anahtari gerekli.
# Opsiyonel anahtarlar: max_model_len, quantization, dtype,
#   gpu_memory_utilization, limit_mm_per_prompt
# ----------------------------------------------------------
MODELS = [
    {"name": "Qwen/Qwen2.5-VL-7B-Instruct"},
    {"name": "OpenGVLab/InternVL2_5-8B"},
    {"name": "llava-hf/llava-onevision-qwen2-7b-ov-hf"},
    {"name": "google/paligemma2-3b-pt-896"},
    {"name": "mistralai/Pixtral-12B-2409"},
    # Yeni model eklemek icin asagiya bir satir ekle:
    # {"name": "herhangi/bir-vlm-modeli"},
    # Ozel parametreler ile ornek:
    # {"name": "meta-llama/Llama-4-Scout-17B-16E-Instruct", "max_model_len": 8192, "dtype": "bfloat16"},
]

# ----------------------------------------------------------
# DRIVE & CIKTI AYARLARI
# Sonuclar Drive'a kaydedilir, boylece Colab kapansa bile kaybolmaz.
# ----------------------------------------------------------
DRIVE_BASE_PATH = "/content/drive/MyDrive/bilsem"              # Drive'daki soru klasoru
OUTPUT_DIR = "/content/drive/MyDrive/bilsem/eval_results"      # Sonuclar Drive'a kaydedilir
CHECKPOINT_FILE = "checkpoint.json"                             # Checkpoint dosya adi (OUTPUT_DIR icinde)

# ----------------------------------------------------------
# INFERENCE AYARLARI
# ----------------------------------------------------------
BATCH_SIZE = 16              # vLLM max_num_seqs
MAX_TOKENS = 512             # Modelin uretecegi max token
TEMPERATURE = 0.0            # Deterministic cevap icin 0
MAX_MODEL_LEN = 4096         # Varsayilan context uzunlugu
MAX_RETRIES = 2              # Hata durumunda tekrar deneme sayisi

# Quantization: None (FP16/BF16), "awq", "gptq", "bitsandbytes"
# Colab T4 (16GB) icin "awq" veya "gptq" onerilir
# Colab A100 (40GB) icin None yeterli
DEFAULT_QUANTIZATION = None

# GPU bellek kullanim orani (0.0 - 1.0)
DEFAULT_GPU_MEMORY_UTILIZATION = 0.90

# ----------------------------------------------------------
# SISTEM PROMPT
# ----------------------------------------------------------
SYSTEM_PROMPT = (
    "Bu bir çoktan seçmeli sınav sorusudur. "
    "Verilen resmi dikkatlice incele ve soruyu çöz. "
    "Cevabını sadece <answer>CEVAP</answer> formatında ver. "
    "CEVAP yalnızca A, B, C, D veya E olmalıdır."
)

# ----------------------------------------------------------
# FILTRELER (opsiyonel, bos liste = hepsi)
# ----------------------------------------------------------
FILTER_EXAMS = []       # ornek: ["Tyt"] veya ["Ayt"] veya ["Tyt", "Ayt"]
FILTER_SUBJECTS = []    # ornek: ["Matematik"] veya ["Fen", "Turkce"]
FILTER_YEARS = []       # ornek: [2024] veya [2024, 2025]

print("Konfigurasyon tamamlandi.")
print(f"  Model sayisi : {len(MODELS)}")
print(f"  Drive yolu   : {DRIVE_BASE_PATH}")
print(f"  Cikti yolu   : {OUTPUT_DIR}")
print(f"  Checkpoint   : {CHECKPOINT_FILE}")

In [ ]:
# ============================================================
# HUCRE 3: IMPORTLAR + DRIVE MOUNT + SORU YUKLEME
# ============================================================

import os
import re
import json
import gc
import base64
import traceback
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from google.colab import drive

# ----------------------------------------------------------
# Drive'i bagla
# ----------------------------------------------------------
drive.mount('/content/drive')

# Cikti klasorunu olustur
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------
# Dosya adi parse fonksiyonu
# ----------------------------------------------------------
def parse_filename(filepath):
    """
    Dosya adindan metadata parse et.
    Format: question-{yil}_{sinav}_{ders}_{soru_no}. answer-{cevap}.jpg
    """
    filename = os.path.basename(filepath)
    pattern = r'question-(\d{4})_(\w+?)_(\w+?)_(\d+)\.\s*answer-([A-EXx])\.(?:jpg|jpeg|png)'
    match = re.match(pattern, filename, re.IGNORECASE)
    if not match:
        return None
    year, exam, subject, question_no, correct_answer = match.groups()
    return {
        "file_path": filepath,
        "filename": filename,
        "year": int(year),
        "exam": exam,
        "subject": subject,
        "question_no": int(question_no),
        "correct_answer": correct_answer.upper(),
    }

# ----------------------------------------------------------
# Soru yukleme fonksiyonu
# ----------------------------------------------------------
def load_questions(base_path):
    """Drive'dan tum soru dosyalarini tara ve metadata parse et."""
    questions = []
    skipped_extra = 0
    skipped_parse = 0

    for root, dirs, files in os.walk(base_path):
        for f in sorted(files):
            if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            filepath = os.path.join(root, f)
            meta = parse_filename(filepath)
            if meta is None:
                skipped_parse += 1
                continue
            # ekstra sorulari atla (cevap X)
            if meta["correct_answer"] == "X":
                skipped_extra += 1
                continue
            questions.append(meta)

    df = pd.DataFrame(questions)
    if len(df) > 0:
        df = df.sort_values(
            ["year", "exam", "subject", "question_no"]
        ).reset_index(drop=True)

    print(f"Yuklenen soru sayisi  : {len(df)}")
    print(f"Atlanan (ekstra/X)    : {skipped_extra}")
    print(f"Atlanan (parse hatasi): {skipped_parse}")
    return df

# ----------------------------------------------------------
# Sorulari yukle
# ----------------------------------------------------------
questions_df = load_questions(DRIVE_BASE_PATH)

# Filtreleri uygula
if FILTER_EXAMS:
    questions_df = questions_df[questions_df["exam"].isin(FILTER_EXAMS)]
if FILTER_SUBJECTS:
    questions_df = questions_df[questions_df["subject"].isin(FILTER_SUBJECTS)]
if FILTER_YEARS:
    questions_df = questions_df[questions_df["year"].isin(FILTER_YEARS)]
questions_df = questions_df.reset_index(drop=True)

# Ozet
print(f"\nFiltre sonrasi toplam: {len(questions_df)} soru")
print(f"\n--- Sinav Dagilimi ---")
print(questions_df.groupby(["year", "exam"]).size().to_string())
print(f"\n--- Ders Dagilimi ---")
print(questions_df.groupby(["year", "exam", "subject"]).size().to_string())

In [ ]:
# ============================================================
# HUCRE 4: INFERENCE MOTORU
# Modelden bagimsiz, vLLM tabanli unified inference fonksiyonu.
# ============================================================

from vllm import LLM, SamplingParams


def image_to_base64(image_path: str) -> str:
    """Resim dosyasini base64 string'e cevir."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def build_conversation(image_path: str, system_prompt: str) -> list[dict]:
    """
    Tek bir soru icin OpenAI-uyumlu chat mesajlari olustur.
    vLLM'in chat() metodu bu formati kabul eder.
    """
    b64 = image_to_base64(image_path)
    data_uri = f"data:image/jpeg;base64,{b64}"

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": data_uri}},
            {"type": "text", "text": "Bu soruyu çöz."},
        ],
    })
    return messages


def run_model(model_config: dict, questions_df, system_prompt: str,
              sampling_params) -> list[dict] | None:
    """
    Tek bir model icin tum sorulari batch olarak isle.
    Herhangi bir vLLM-destekli vision modeli icin calisir.
    Basarisiz olursa None doner.
    """
    model_name = model_config["name"]

    # Model parametrelerini config'den veya varsayilandan al
    max_model_len = model_config.get("max_model_len", MAX_MODEL_LEN)
    quantization = model_config.get("quantization", DEFAULT_QUANTIZATION)
    dtype = model_config.get("dtype", "auto")
    gpu_mem = model_config.get("gpu_memory_utilization",
                               DEFAULT_GPU_MEMORY_UTILIZATION)
    mm_limit = model_config.get("limit_mm_per_prompt", {"image": 1})

    print(f"\n{'=' * 60}")
    print(f"Model yukleniyor: {model_name}")
    print(f"  max_model_len={max_model_len}  quantization={quantization}")
    print(f"  dtype={dtype}  gpu_memory_utilization={gpu_mem}")
    print(f"{'=' * 60}")

    # ---- Model yukle ----
    try:
        llm = LLM(
            model=model_name,
            trust_remote_code=True,
            max_model_len=max_model_len,
            max_num_seqs=BATCH_SIZE,
            gpu_memory_utilization=gpu_mem,
            quantization=quantization,
            dtype=dtype,
            limit_mm_per_prompt=mm_limit,
        )
    except Exception as e:
        print(f"[HATA] Model yuklenemedi: {model_name}")
        print(f"  {e}")
        traceback.print_exc()
        return None

    # ---- Chat mesajlarini hazirla ----
    total = len(questions_df)
    print(f"Toplam {total} soru icin chat mesajlari hazirlaniyor...")

    all_conversations = []
    for _, row in tqdm(questions_df.iterrows(), total=total,
                       desc="Mesaj hazirlama"):
        conv = build_conversation(row["file_path"], system_prompt)
        all_conversations.append(conv)

    # ---- Batch inference ----
    print("Batch inference basliyor...")
    try:
        outputs = llm.chat(
            messages=all_conversations,
            sampling_params=sampling_params,
            use_tqdm=True,
        )
    except Exception as e:
        print(f"[HATA] Batch inference basarisiz: {e}")
        traceback.print_exc()
        del llm
        gc.collect()
        torch.cuda.empty_cache()
        return None

    # ---- Sonuclari topla ----
    timestamp = datetime.now().isoformat()
    results = []
    for output, (_, row) in zip(outputs, questions_df.iterrows()):
        raw_text = output.outputs[0].text
        results.append({
            "model": model_name,
            "timestamp": timestamp,
            "file_path": row["file_path"],
            "filename": row["filename"],
            "year": int(row["year"]),
            "exam": row["exam"],
            "subject": row["subject"],
            "question_no": int(row["question_no"]),
            "correct_answer": row["correct_answer"],
            "raw_response": raw_text,
        })

    # ---- Bellegi temizle ----
    print(f"Model bellekten siliniyor: {model_name}")
    del llm
    gc.collect()
    torch.cuda.empty_cache()

    print(f"[OK] {model_name} tamamlandi — {len(results)} soru islendi.")
    return results

In [ ]:
# ============================================================
# HUCRE 5: CEVAP PARSE + DEGERLENDIRME + LOGLAMA
# ============================================================


def parse_answer(raw_response: str) -> str:
    """
    Model cevabindan <answer>X</answer> formatini parse et.
    Parse edilemezse 'INVALID' doner.
    """
    if not raw_response:
        return "INVALID"

    # Birincil: <answer>X</answer> tagi
    pattern = r'<answer>\s*([A-Ea-e])\s*</answer>'
    match = re.search(pattern, raw_response)
    if match:
        return match.group(1).upper()

    # Ikincil: <answer>X<answer> (kapanma tag'i yanlis yazilmis)
    pattern2 = r'<answer>\s*([A-Ea-e])\s*</?answer>'
    match2 = re.search(pattern2, raw_response)
    if match2:
        return match2.group(1).upper()

    # Ucuncul fallback: Cevap cok kisaysa ve tek gecerli harf ise
    clean = raw_response.strip()
    if len(clean) == 1 and clean.upper() in "ABCDE":
        return clean.upper()

    # Son fallback: Cevap icinde "Cevap: X" veya "Answer: X" gibi pattern
    pattern3 = r'(?:cevap|answer|yanıt|doğru\s*(?:cevap|şık))\s*[:=]?\s*([A-Ea-e])\b'
    match3 = re.search(pattern3, raw_response, re.IGNORECASE)
    if match3:
        return match3.group(1).upper()

    return "INVALID"


def evaluate_results(results: list[dict]) -> list[dict]:
    """Her sonuc icin model_answer, is_correct, is_valid alanlarini ekle."""
    for r in results:
        r["model_answer"] = parse_answer(r["raw_response"])
        r["is_correct"] = (r["model_answer"] == r["correct_answer"])
        r["is_valid"] = (r["model_answer"] != "INVALID")
    return results


# ----------------------------------------------------------
# Checkpoint fonksiyonlari
# Colab kesilse bile kaldigi yerden devam edebilmek icin
# her model bittikten sonra checkpoint Drive'a yazilir.
# ----------------------------------------------------------

def _checkpoint_path() -> str:
    """Checkpoint dosyasinin tam yolunu dondur."""
    return os.path.join(OUTPUT_DIR, CHECKPOINT_FILE)


def load_checkpoint() -> dict:
    """
    Mevcut checkpoint dosyasini oku.
    Yoksa bos bir checkpoint dondur.
    Dondurulen format:
      {
        "completed_models": ["model/a", "model/b"],
        "results": [ ... tum sonuc dict'leri ... ]
      }
    """
    cp_path = _checkpoint_path()
    if os.path.exists(cp_path):
        try:
            with open(cp_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            n_models = len(data.get("completed_models", []))
            n_results = len(data.get("results", []))
            print(f"[CHECKPOINT] Mevcut checkpoint yuklendi: "
                  f"{n_models} model tamamlanmis, {n_results} sonuc.")
            return data
        except Exception as e:
            print(f"[UYARI] Checkpoint okunamadi, sifirdan baslanacak: {e}")
    return {"completed_models": [], "results": []}


def save_checkpoint(checkpoint: dict) -> None:
    """
    Checkpoint'i Drive'a kaydet.
    Her model bittikten sonra cagirilir — anlik kayit.
    """
    cp_path = _checkpoint_path()
    with open(cp_path, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)
    n_models = len(checkpoint.get("completed_models", []))
    n_results = len(checkpoint.get("results", []))
    print(f"  [CHECKPOINT] Kaydedildi -> {n_models} model, {n_results} sonuc")


def get_completed_models(checkpoint: dict) -> set:
    """Checkpoint'ten tamamlanmis model isimlerini dondur."""
    return set(checkpoint.get("completed_models", []))


# ----------------------------------------------------------
# Loglama fonksiyonlari
# ----------------------------------------------------------

def save_model_results(results: list[dict], model_name: str,
                       output_dir: str) -> str:
    """Tek bir modelin sonuclarini ayrica JSON dosyasina kaydet."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_name = model_name.replace("/", "__")

    json_path = os.path.join(output_dir, f"results_{safe_name}_{ts}.json")
    payload = {
        "model": model_name,
        "timestamp": ts,
        "total_questions": len(results),
        "correct": sum(1 for r in results if r.get("is_correct")),
        "invalid": sum(1 for r in results if not r.get("is_valid", True)),
        "accuracy_pct": round(
            sum(1 for r in results if r.get("is_correct")) / len(results) * 100, 2
        ) if results else 0.0,
        "results": results,
    }
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"  JSON kaydedildi: {json_path}")
    return json_path


def save_all_results_csv(all_results: list[dict],
                         output_dir: str) -> str | None:
    """Tum modellerin sonuclarini tek bir CSV'ye kaydet."""
    if not all_results:
        print("Kaydedilecek sonuc yok.")
        return None

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = os.path.join(output_dir, f"all_results_{ts}.csv")

    df = pd.DataFrame(all_results)
    # raw_response uzun olabilir — kisaltilmis surumunu de ekle
    if "raw_response" in df.columns:
        df["raw_response_preview"] = df["raw_response"].str[:300]

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"  Birlesik CSV kaydedildi: {csv_path}")
    return csv_path


print("Cevap parse, degerlendirme, checkpoint ve loglama fonksiyonlari hazir.")

In [ ]:
# ============================================================
# HUCRE 6: RAPORLAMA
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate


def generate_report(all_results: list[dict]) -> None:
    """Tum model sonuclarindan detayli eval raporu olustur."""
    if not all_results:
        print("Raporlanacak sonuc yok.")
        return

    df = pd.DataFrame(all_results)

    print("=" * 70)
    print("              VLM SINAV EVAL SONUC RAPORU")
    print(f"              {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)

    # ---- 1. Model Bazli Ozet ----
    print("\n\n[1] MODEL BAZLI OZET")
    print("-" * 70)
    model_summary = df.groupby("model").agg(
        Toplam=("is_correct", "count"),
        Dogru=("is_correct", "sum"),
        Gecersiz=("is_valid", lambda x: (~x).sum()),
    ).reset_index()
    model_summary["Yanlis"] = (
        model_summary["Toplam"] - model_summary["Dogru"] - model_summary["Gecersiz"]
    )
    model_summary["Accuracy (%)"] = (
        model_summary["Dogru"] / model_summary["Toplam"] * 100
    ).round(2)
    model_summary = model_summary.sort_values("Accuracy (%)", ascending=False)
    print(tabulate(model_summary, headers="keys", tablefmt="grid",
                   showindex=False))

    # ---- 2. Ders Bazli Kirilim ----
    print("\n\n[2] DERS BAZLI ACCURACY (%)")
    print("-" * 70)
    subj_pivot = df.pivot_table(
        index="model", columns="subject",
        values="is_correct", aggfunc="mean",
    ) * 100
    subj_pivot = subj_pivot.round(2)
    print(tabulate(subj_pivot, headers="keys", tablefmt="grid"))

    # ---- 3. Sinav Bazli Kirilim ----
    print("\n\n[3] SINAV BAZLI ACCURACY (%)")
    print("-" * 70)
    exam_pivot = df.pivot_table(
        index="model", columns="exam",
        values="is_correct", aggfunc="mean",
    ) * 100
    exam_pivot = exam_pivot.round(2)
    print(tabulate(exam_pivot, headers="keys", tablefmt="grid"))

    # ---- 4. Yil Bazli Kirilim ----
    print("\n\n[4] YIL BAZLI ACCURACY (%)")
    print("-" * 70)
    year_pivot = df.pivot_table(
        index="model", columns="year",
        values="is_correct", aggfunc="mean",
    ) * 100
    year_pivot = year_pivot.round(2)
    print(tabulate(year_pivot, headers="keys", tablefmt="grid"))

    # ---- 5. Detayli Kirilim (Sinav x Ders x Model) ----
    print("\n\n[5] DETAYLI KIRILIM: MODEL x SINAV x DERS")
    print("-" * 70)
    detail = df.groupby(["model", "year", "exam", "subject"]).agg(
        Toplam=("is_correct", "count"),
        Dogru=("is_correct", "sum"),
    ).reset_index()
    detail["Accuracy (%)"] = (
        detail["Dogru"] / detail["Toplam"] * 100
    ).round(2)
    print(tabulate(detail, headers="keys", tablefmt="grid", showindex=False))

    # ---- 6. En Zor Sorular ----
    print("\n\n[6] EN ZOR SORULAR (Hicbir modelin dogru cevaplayamadigi)")
    print("-" * 70)
    q_correct = df.groupby(
        ["year", "exam", "subject", "question_no", "correct_answer"]
    ).agg(
        models_correct=("is_correct", "sum"),
        total_models=("is_correct", "count"),
    ).reset_index()
    hardest = q_correct[q_correct["models_correct"] == 0]
    if len(hardest) > 0:
        print(tabulate(hardest, headers="keys", tablefmt="grid",
                       showindex=False))
    else:
        print("  Tum sorularda en az bir model dogru cevap verdi.")

    # ---- 7. Model Cevap Dagilimi ----
    print("\n\n[7] MODEL CEVAP DAGILIMI (hangi sikki ne kadar secti)")
    print("-" * 70)
    for mname in sorted(df["model"].unique()):
        mdf = df[df["model"] == mname]
        dist = mdf["model_answer"].value_counts().sort_index()
        total_valid = mdf["is_valid"].sum()
        total_correct = mdf["is_correct"].sum()
        print(f"\n  {mname}")
        print(f"  Gecerli cevap: {total_valid}/{len(mdf)}  "
              f"Dogru: {total_correct}/{len(mdf)}")
        for ans, cnt in dist.items():
            bar = "█" * int(cnt / len(mdf) * 40)
            print(f"    {ans:>7s}: {cnt:4d}  {bar}")

    # ---- 8. Grafikler ----
    try:
        _plot_results(df)
    except Exception as e:
        print(f"\n[UYARI] Grafik olusturulamadi: {e}")


def _plot_results(df: pd.DataFrame) -> None:
    """Sonuclari matplotlib ile gorsellestir."""
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(18, 13))
    fig.suptitle("VLM Sinav Eval Sonuclari", fontsize=16, fontweight="bold")

    short = lambda s: s.split("/")[-1] if "/" in s else s

    # 1 — Model accuracy
    ax = axes[0, 0]
    macc = df.groupby("model")["is_correct"].mean().sort_values() * 100
    colors = sns.color_palette("viridis", len(macc))
    ax.barh([short(n) for n in macc.index], macc.values, color=colors)
    ax.set_xlabel("Accuracy (%)")
    ax.set_title("Model Bazli Accuracy")
    for i, v in enumerate(macc.values):
        ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9)

    # 2 — Ders bazli
    ax = axes[0, 1]
    sacc = df.groupby(["model", "subject"])["is_correct"].mean().unstack() * 100
    sacc.index = [short(n) for n in sacc.index]
    sacc.plot(kind="bar", ax=ax, rot=30)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Ders Bazli Accuracy")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)

    # 3 — Sinav bazli
    ax = axes[1, 0]
    eacc = df.groupby(["model", "exam"])["is_correct"].mean().unstack() * 100
    eacc.index = [short(n) for n in eacc.index]
    eacc.plot(kind="bar", ax=ax, rot=30)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Sinav Bazli Accuracy")

    # 4 — Yil bazli
    ax = axes[1, 1]
    yacc = df.groupby(["model", "year"])["is_correct"].mean().unstack() * 100
    yacc.index = [short(n) for n in yacc.index]
    yacc.plot(kind="bar", ax=ax, rot=30)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Yil Bazli Accuracy")

    plt.tight_layout()
    plot_path = os.path.join(
        OUTPUT_DIR,
        f"eval_plots_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png",
    )
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Grafik kaydedildi: {plot_path}")


print("Raporlama fonksiyonlari hazir.")

In [ ]:
# ============================================================
# HUCRE 7: ANA CALISMA DONGUSU  (resume destekli)
#
# Akis:
#   1. Checkpoint'i oku (varsa onceki sonuclari yukle)
#   2. Tamamlanmis modelleri atla
#   3. Her model bittikten sonra checkpoint'i anlik Drive'a yaz
#   4. Colab kesilirse, hucreyi tekrar calistirmak yeterli
# ============================================================

# ---- 1. Checkpoint yukle ----
checkpoint = load_checkpoint()
completed_models = get_completed_models(checkpoint)
all_results = list(checkpoint.get("results", []))   # onceki sonuclari devral

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

total_models = len(MODELS)
skipped = 0

for model_idx, model_config in enumerate(MODELS, 1):
    model_name = model_config["name"]

    # ---- 2. Zaten tamamlanmis mi? ----
    if model_name in completed_models:
        skipped += 1
        print(f"\n[{model_idx}/{total_models}] ATLANDI (zaten tamamlanmis): "
              f"{model_name}")
        continue

    print(f"\n{'#' * 70}")
    print(f"# [{model_idx}/{total_models}] MODEL: {model_name}")
    print(f"{'#' * 70}")

    success = False
    for attempt in range(MAX_RETRIES + 1):
        if attempt > 0:
            print(f"\n  Yeniden deneme {attempt}/{MAX_RETRIES}...")

        try:
            results = run_model(
                model_config, questions_df, SYSTEM_PROMPT, sampling_params
            )

            if results:
                # Cevaplari parse et ve degerlendir
                results = evaluate_results(results)

                # Model JSON'ini ayrica kaydet (Drive'a)
                save_model_results(results, model_name, OUTPUT_DIR)

                # Genel listeye ekle
                all_results.extend(results)

                # ---- 3. Checkpoint'i anlik guncelle ve Drive'a yaz ----
                checkpoint["results"] = all_results
                checkpoint["completed_models"] = list(
                    completed_models | {model_name}
                )
                save_checkpoint(checkpoint)
                completed_models.add(model_name)

                # Hizli ozet
                correct = sum(1 for r in results if r["is_correct"])
                invalid = sum(1 for r in results if not r["is_valid"])
                total_q = len(results)
                acc = correct / total_q * 100 if total_q > 0 else 0
                print(f"\n  >>> OZET: {correct}/{total_q} dogru "
                      f"({acc:.1f}%), {invalid} gecersiz cevap")
                success = True
                break
            else:
                print(f"  Model sonuc dondurmedi: {model_name}")

        except Exception as e:
            print(f"  [HATA] Deneme {attempt + 1}: {e}")
            traceback.print_exc()
            gc.collect()
            torch.cuda.empty_cache()

    if not success:
        print(f"\n  !!! MODEL BASARISIZ: {model_name} "
              f"— tum denemeler tukendi.\n")

# ----------------------------------------------------------
# Tum sonuclari birlestirip kaydet ve raporla
# ----------------------------------------------------------
if skipped > 0:
    print(f"\n  ({skipped} model onceden tamamlanmisti, atlandi.)")

if all_results:
    print(f"\n{'=' * 70}")
    print(f"TAMAMLANDI — Toplam {len(all_results)} sonuc mevcut.")
    print(f"{'=' * 70}")

    # CSV kaydet (Drive'a)
    save_all_results_csv(all_results, OUTPUT_DIR)

    # Detayli rapor
    generate_report(all_results)
else:
    print("\nHicbir model basarili sonuc dondurmedi.")